## Setup

In [23]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from patsy import dmatrices

In [24]:
df = pd.read_csv('/Users/emmiekao/Downloads/datasci112_final (1).csv')

In [25]:
df

,FIPS,County,"Convenience stores/1,000 population","Fast-food restaurants/1,000 population","Grocery stores/1,000 population",Median household income,"% Households, no car & low access to store",% Population low income & low access to store,% Population with low access to store,"SNAP-authorized stores/1,000 population","Supercenters/1,000 population",% Adults with Obesity,% Adults with Diabetes,Retail Food Environment Index,Cancer Incidence Rate
0,6001,Alameda,0.197314,0.827156,0.219572,108971.0,0.342808,1.117399,8.083675,0.545731,0.016242,23.5,9.1,4.344388,31.7
1,6003,Alpine,NaN,2.680965,NaN,87570.0,6.022490,16.785139,56.969242,NaN,NaN,29.9,9.9,NaN,NaN
2,6005,Amador,0.324327,0.573809,0.299379,68159.0,3.163143,0.095632,0.174187,0.763980,NaN,29.4,9.0,NaN,39.3
3,6007,Butte,0.329034,0.737976,0.188019,62982.0,1.979342,6.450867,19.113434,0.926945,0.018802,31.7,9.5,5.159091,42.5
4,6009,Calaveras,0.410296,0.604647,0.259135,68298.0,1.547478,5.989503,14.736117,0.985243,NaN,30.1,8.7,NaN,38.4
5,6011,Colusa,0.556638,0.417478,0.324705,60725.0,2.258987,12.356255,39.329746,1.283756,NaN,32.3,11.8,NaN,40.1
6,6013,Contra Costa,0.200463,0.678623,0.194388,110595.0,0.766959,2.867471,23.228312,0.542727,0.013885,23.9,9.4,4.220833,34.2
7,6015,Del Norte,0.178776,0.321796,0.143021,48108.0,3.967303,14.445117,34.407192,0.983177,NaN,33.4,10.9,NaN,24.5
8,6017,El Dorado,0.285085,0.673837,0.171051,87689.0,1.906889,4.258567,25.222891,0.636368,0.020733,26.0,7.9,5.000000,35.6
9,6019,Fresno,0.298726,0.731329,0.252768,63140.0,0.889459,5.349311,13.327069,0.916412,0.019982,37.6,13.5,3.776557,33.1


## Prep dataframe

Rename columns

In [26]:
df.columns = df.columns.str.strip()

rename_map = {
    "Retail Food Environment Index": "rfei",
    "Median household income": "income",
    "% Adults with Obesity": "obesity",
    "% Adults with Diabetes": "diabetes",
    "Cancer Incidence Rate": "cancer",
    "% Households, no car & low access to store": "no_car_low_access",
    "% Population low income & low access to store": "low_income_low_access",
    "% Population with low access to store": "low_access",
    "Convenience stores/1,000 population": "convenience",
    "Fast-food restaurants/1,000 population": "fast_food",
    "Grocery stores/1,000 population": "grocery",
    "SNAP-authorized stores/1,000 population": "snap",
    "Supercenters/1,000 population": "supercenters"
}

df = df.rename(columns=rename_map)

needed = [
    "income", "rfei",
    "no_car_low_access", "low_income_low_access", "low_access",
    "convenience", "fast_food", "grocery", "snap", "supercenters",
    "obesity", "diabetes", "cancer"
]

df = df[needed].apply(pd.to_numeric, errors="coerce").dropna().copy()

There are two categories of variable in our dataset: `access` variables (describing food accessibility in a county) and `store` variables (describing the amount of stores in a county). 

We will attempt two strategies to create linear regressions. The first is simply using all variables as individuals, and the second is Principle Component Analysis (PCA), where we condense each group of variables into one while attempting to retain the most important information.

Below, we create these condensed variables.

In [27]:
access_vars = ["no_car_low_access", "low_income_low_access", "low_access"]
store_vars = ["convenience", "fast_food", "grocery", "snap", "supercenters"]
all_non_rfei_vars = access_vars + store_vars

def make_pc1(data, cols, new_name):
    scaler = StandardScaler()
    Z = scaler.fit_transform(data[cols])

    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(Z).ravel()

    loadings = pd.Series(pca.components_[0], index=cols)

    if loadings.mean() < 0:
        pc1 *= -1
        loadings *= -1

    out = {
        "scores": pc1,
        "explained_variance": pca.explained_variance_ratio_[0],
        "loadings": loadings.sort_values()
    }
    return out

access_info = make_pc1(df, access_vars, "access_pc1")
store_info = make_pc1(df, store_vars, "store_pc1")
combined_info = make_pc1(df, all_non_rfei_vars, "combined_pc1")

df["access_pc1"] = access_info["scores"]
df["store_pc1"] = store_info["scores"]
df["combined_pc1"] = combined_info["scores"]

print("Access PC1 explained variance:", access_info["explained_variance"])
print(access_info["loadings"], "\n")

print("Store PC1 explained variance:", store_info["explained_variance"])
print(store_info["loadings"], "\n")

print("Combined PC1 explained variance:", combined_info["explained_variance"])
print(combined_info["loadings"])

Access PC1 explained variance: 0.7950938869810505
low_access               0.553703
no_car_low_access        0.568862
low_income_low_access    0.608120
dtype: float64 

Store PC1 explained variance: 0.42895670764654803
fast_food      -0.519114
grocery        -0.025683
supercenters    0.025955
snap            0.581756
convenience     0.625098
dtype: float64 

Combined PC1 explained variance: 0.4878298090423431
fast_food               -0.344809
grocery                 -0.138636
supercenters            -0.020501
low_access               0.343684
snap                     0.375504
convenience              0.425898
no_car_low_access        0.458577
low_income_low_access    0.458979
dtype: float64


## Helper functions and formulas

Create helper functions for models

In [28]:
def cv_rmse(formula, data, n_splits=5):
    y, X = dmatrices(formula, data=data, return_type="dataframe")
    y = np.ravel(y)

    pipe = Pipeline([
        ("scale", StandardScaler(with_mean=False)),
        ("lr", LinearRegression())
    ])

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )
    return -scores.mean()

def partial_r2(reduced_model, full_model):
    return (full_model.rsquared - reduced_model.rsquared) / (1 - reduced_model.rsquared)

def fit_formula(formula, data):
    return smf.ols(formula, data=data).fit()

We want to test essentially all possible outcomes. We use `income` as a baseline, since we know that it is already strongly correlated with the health outcomes.

In [29]:
outcomes = ["obesity", "diabetes", "cancer"]

def get_formulas(y):
    formulas = {
        "baseline": f"{y} ~ income",

        "rfei_linear": f"{y} ~ rfei",
        "rfei_quadratic": f"{y} ~ rfei + I(rfei**2)",

        "access_only_pc": f"{y} ~ access_pc1",
        "store_only_pc": f"{y} ~ store_pc1",
        "combined_only_pc": f"{y} ~ combined_pc1",

        "rfei_plus_access_linear": f"{y} ~ access_pc1 + rfei",
        "rfei_plus_access_quadratic": f"{y} ~ access_pc1 + rfei + I(rfei**2)",

        "rfei_plus_store_linear": f"{y} ~ store_pc1 + rfei",
        "rfei_plus_store_quadratic": f"{y} ~ store_pc1 + rfei + I(rfei**2)",

        "controls_separate_pc": f"{y} ~ access_pc1 + store_pc1",
        "rfei_plus_separate_linear": f"{y} ~ access_pc1 + store_pc1 + rfei",
        "rfei_plus_separate_quadratic": f"{y} ~ access_pc1 + store_pc1 + rfei + I(rfei**2)",

        "rfei_plus_combined_linear": f"{y} ~ combined_pc1 + rfei",
        "rfei_plus_combined_quadratic": f"{y} ~ combined_pc1 + rfei + I(rfei**2)",

        "raw_access": f"{y} ~ no_car_low_access + low_income_low_access + low_access",
        "raw_store": f"{y} ~ convenience + fast_food + grocery + snap + supercenters",
        "raw_all": f"{y} ~ no_car_low_access + low_income_low_access + low_access + convenience + fast_food + grocery + snap + supercenters"
    }
    return formulas

## Fit Models and Assess

In [48]:
results = []
models = {}

for y in outcomes:
    formulas = get_formulas(y)
    fitted = {}

    for model_name, formula in formulas.items():
        model = fit_formula(formula, df)
        fitted[model_name] = model
        models[(y, model_name)] = model

        results.append({
            "outcome": y,
            "model": model_name,
            "formula": formula,
            "n_params": int(model.df_model + 1),
            "r2": model.rsquared,
            "cv_rmse": cv_rmse(formula, df)
        })

results_df = pd.DataFrame(results)
results_df.head()

,outcome,model,formula,n_params,r2,cv_rmse
0,obesity,baseline,obesity ~ income,2,0.696579,2.870767
1,obesity,rfei_linear,obesity ~ rfei,2,0.006371,5.119239
2,obesity,rfei_quadratic,obesity ~ rfei + I(rfei**2),3,0.039556,5.043944
3,obesity,access_only_pc,obesity ~ access_pc1,2,0.388775,4.219348
4,obesity,store_only_pc,obesity ~ store_pc1,2,0.483721,3.969381


In [49]:
results_df.to_csv("full_model_results.csv") # for visualizations

### Compare models

In [35]:
# best by CV RMSE within each outcome
best_by_rmse = (
    results_df
    .sort_values(["outcome", "cv_rmse", "r2"], ascending=[True, True, False])
    .groupby("outcome", as_index=False)
    .head(8)
)

best_by_rmse[["outcome", "model", "r2", "cv_rmse"]]

,outcome,model,r2,cv_rmse
51,cancer,raw_access,0.372482,2.384731
41,cancer,combined_only_pc,0.171439,2.468861
40,cancer,store_only_pc,0.133940,2.470346
46,cancer,controls_separate_pc,0.157381,2.507929
36,cancer,baseline,0.131084,2.556303
39,cancer,access_only_pc,0.131951,2.583162
50,cancer,rfei_plus_combined_quadratic,0.193229,2.674513
49,cancer,rfei_plus_combined_linear,0.172236,2.711230
33,diabetes,raw_access,0.707878,1.146671
18,diabetes,baseline,0.465824,1.462990


In [36]:
# best by adjusted R^2 within each outcome
best_by_r2 = (
    results_df
    .sort_values(["outcome", "r2", "cv_rmse"], ascending=[True, False, True])
    .groupby("outcome", as_index=False)
    .head(8)
)

best_by_r2[["outcome", "model", "r2", "cv_rmse"]]

,outcome,model,r2,cv_rmse
53,cancer,raw_all,0.605732,2.801748
51,cancer,raw_access,0.372482,2.384731
52,cancer,raw_store,0.327636,2.719932
50,cancer,rfei_plus_combined_quadratic,0.193229,2.674513
48,cancer,rfei_plus_separate_quadratic,0.183641,2.805844
49,cancer,rfei_plus_combined_linear,0.172236,2.711230
45,cancer,rfei_plus_store_quadratic,0.172038,2.732678
41,cancer,combined_only_pc,0.171439,2.468861
35,diabetes,raw_all,0.797597,1.508448
33,diabetes,raw_access,0.707878,1.146671


## Sort results by health outcome

In [37]:
obesity = results_df[results_df["outcome"]=="obesity"]

In [38]:
diabetes = results_df[results_df["outcome"]=="diabetes"]

In [39]:
cancer = results_df[results_df["outcome"]=="cancer"]

In [40]:
obesity = obesity.sort_values(by="r2", ascending=False)
obesity

,outcome,model,formula,n_params,r2,cv_rmse
17,obesity,raw_all,obesity ~ no_car_low_access + low_income_low_a...,9,0.835772,3.240952
0,obesity,baseline,obesity ~ income,2,0.696579,2.870767
15,obesity,raw_access,obesity ~ no_car_low_access + low_income_low_a...,4,0.673942,3.256631
16,obesity,raw_store,obesity ~ convenience + fast_food + grocery + ...,6,0.645985,4.015784
12,obesity,rfei_plus_separate_quadratic,obesity ~ access_pc1 + store_pc1 + rfei + I(rf...,5,0.532252,4.036089
11,obesity,rfei_plus_separate_linear,obesity ~ access_pc1 + store_pc1 + rfei,4,0.531359,3.981178
10,obesity,controls_separate_pc,obesity ~ access_pc1 + store_pc1,3,0.523210,3.846759
14,obesity,rfei_plus_combined_quadratic,obesity ~ combined_pc1 + rfei + I(rfei**2),4,0.521317,3.992228
13,obesity,rfei_plus_combined_linear,obesity ~ combined_pc1 + rfei,3,0.519788,3.942695
5,obesity,combined_only_pc,obesity ~ combined_pc1,2,0.519544,3.786437


In [41]:
diabetes = diabetes.sort_values(by="r2", ascending=False)
diabetes

,outcome,model,formula,n_params,r2,cv_rmse
35,diabetes,raw_all,diabetes ~ no_car_low_access + low_income_low_...,9,0.797597,1.508448
33,diabetes,raw_access,diabetes ~ no_car_low_access + low_income_low_...,4,0.707878,1.146671
18,diabetes,baseline,diabetes ~ income,2,0.465824,1.462990
34,diabetes,raw_store,diabetes ~ convenience + fast_food + grocery +...,6,0.360862,1.901009
30,diabetes,rfei_plus_separate_quadratic,diabetes ~ access_pc1 + store_pc1 + rfei + I(r...,5,0.262112,1.883042
29,diabetes,rfei_plus_separate_linear,diabetes ~ access_pc1 + store_pc1 + rfei,4,0.262066,1.778697
27,diabetes,rfei_plus_store_quadratic,diabetes ~ store_pc1 + rfei + I(rfei**2),4,0.253553,1.882374
26,diabetes,rfei_plus_store_linear,diabetes ~ store_pc1 + rfei,3,0.253411,1.774558
28,diabetes,controls_separate_pc,diabetes ~ access_pc1 + store_pc1,3,0.235393,1.781403
22,diabetes,store_only_pc,diabetes ~ store_pc1,2,0.235199,1.775419


In [42]:
cancer = cancer.sort_values(by="r2", ascending=False)
cancer

,outcome,model,formula,n_params,r2,cv_rmse
53,cancer,raw_all,cancer ~ no_car_low_access + low_income_low_ac...,9,0.605732,2.801748
51,cancer,raw_access,cancer ~ no_car_low_access + low_income_low_ac...,4,0.372482,2.384731
52,cancer,raw_store,cancer ~ convenience + fast_food + grocery + s...,6,0.327636,2.719932
50,cancer,rfei_plus_combined_quadratic,cancer ~ combined_pc1 + rfei + I(rfei**2),4,0.193229,2.674513
48,cancer,rfei_plus_separate_quadratic,cancer ~ access_pc1 + store_pc1 + rfei + I(rfe...,5,0.183641,2.805844
49,cancer,rfei_plus_combined_linear,cancer ~ combined_pc1 + rfei,3,0.172236,2.711230
45,cancer,rfei_plus_store_quadratic,cancer ~ store_pc1 + rfei + I(rfei**2),4,0.172038,2.732678
41,cancer,combined_only_pc,cancer ~ combined_pc1,2,0.171439,2.468861
43,cancer,rfei_plus_access_quadratic,cancer ~ access_pc1 + rfei + I(rfei**2),4,0.163580,2.735915
47,cancer,rfei_plus_separate_linear,cancer ~ access_pc1 + store_pc1 + rfei,4,0.161887,2.858335


## Train Models Based on SNAP

In [43]:
results = []
for y in outcomes:
    formulas = {
        "with_snap": f"{y} ~ convenience + fast_food + grocery + snap + supercenters",
        "without_snap": f"{y} ~ convenience + fast_food + grocery + supercenters"
    }

    for model_name, formula in formulas.items():
        model = fit_formula(formula, df)
        models[(y, model_name)] = model

        results.append({
            "outcome": y,
            "model": model_name,
            "formula": formula,
            "n_params": int(model.df_model + 1),
            "r2": model.rsquared,
            "cv_rmse": cv_rmse(formula, df)
        })

results_df = pd.DataFrame(results)
results_df[["outcome", "model", "r2", "cv_rmse"]]

,outcome,model,r2,cv_rmse
0,obesity,with_snap,0.645985,4.015784
1,obesity,without_snap,0.544610,3.983172
2,diabetes,with_snap,0.360862,1.901009
3,diabetes,without_snap,0.196291,2.016048
4,cancer,with_snap,0.327636,2.719932
5,cancer,without_snap,0.195412,3.006018


In [44]:
results_df.to_csv("store_snap.csv")

In [45]:
results = []
for y in outcomes:
    formulas = {
        "only_snap": f"{y} ~ snap",
        "only_rfei": f"{y} ~ rfei"
    }

    for model_name, formula in formulas.items():
        model = fit_formula(formula, df)
        models[(y, model_name)] = model

        results.append({
            "outcome": y,
            "model": model_name,
            "formula": formula,
            "n_params": int(model.df_model + 1),
            "r2": model.rsquared,
            "cv_rmse": cv_rmse(formula, df)
        })

results_df = pd.DataFrame(results)
results_df[["outcome", "model", "r2", "cv_rmse"]]

,outcome,model,r2,cv_rmse
0,obesity,only_snap,0.369309,4.199055
1,obesity,only_rfei,0.006371,5.119239
2,diabetes,only_snap,0.262451,1.814165
3,diabetes,only_rfei,0.004511,1.992230
4,cancer,only_snap,0.167077,2.515679
5,cancer,only_rfei,0.006841,2.973417


In [46]:
results_df.to_csv("only_snap.csv")